# Atelier 2 — Google Earth Engine : analyseur pluviométrique interactif
### Dessinez une zone, choisissez une période, cliquez « Analyser »

**Public** : statisticiens, économistes et data scientists du HCP
**Durée indicative** : 3 h
**Pré-requis** : Python de base, `pandas`, notions de SIG

---

## L'outil que vous allez construire

Une petite application dans le notebook :

1. Une **carte** sur laquelle vous **dessinez** votre zone d'étude (rectangle ou polygone).
2. Deux **sélecteurs de date** — début et fin de période.
3. Un bouton **« Analyser »**.

En sortie :

- le **cumul** pluviométrique sur la zone et la période ;
- la **moyenne**, le **maximum** et le **minimum** — sous leurs deux formes, spatiale et temporelle ;
- la série journalière, la courbe de cumul et le profil mensuel ;
- une carte du cumul ;
- **l'estimation du volume d'images source** qu'il aurait fallu télécharger pour faire
  le même calcul localement.

Ce dernier point n'est pas un gadget. C'est l'argument central : Earth Engine ne vous envoie pas
les données, **il envoie votre calcul aux données**. La différence se compte en ordres de grandeur.

---

## Objectifs pédagogiques

1. Configurer et authentifier Earth Engine en Python.
2. Comprendre la frontière **serveur / client**, la principale source d'erreurs sous GEE.
3. Maîtriser `reduceRegion` et les **réducteurs combinés**.
4. Distinguer les **deux ordres de réduction** — spatial puis temporel, ou l'inverse —
   qui ne donnent pas le même résultat et ne répondent pas à la même question.
5. Construire une interface `ipywidgets` + `geemap` réutilisable.
6. Quantifier le volume de données mobilisé, et savoir l'expliquer à une direction.

---

## 1. Installation, authentification, initialisation

### Prérequis administratifs
1. Un compte Google.
2. Une inscription sur https://earthengine.google.com/ (gratuit pour la recherche et l'usage
   non commercial ; les organismes doivent rattacher un **projet Google Cloud**).
3. Créer un projet Cloud et y **activer l'API Earth Engine**.
4. Noter l'identifiant du projet (ex. `hcp-geospatial-2026`).

> Pour une exécution non interactive (ordonnanceur, production), on utilise un **compte de
> service** : voir le module 9.

In [ ]:
# %pip install earthengine-api geemap ipywidgets pandas matplotlib

import ee, geemap, ipywidgets as widgets
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from datetime import date, datetime, timezone
from IPython.display import display, clear_output

print("earthengine-api :", ee.__version__)
print("geemap          :", geemap.__version__)

earthengine-api : 1.7.35
geemap          : 0.38.3


In [ ]:
PROJET = "ambient-glazing-333820"   # <<< À REMPLACER

try:
    ee.Initialize(project=PROJET)
    print("✅ Earth Engine initialisé")
except Exception:
    print("Authentification nécessaire — une page navigateur va s'ouvrir.")
    ee.Authenticate()          # une seule fois par machine
    ee.Initialize(project=PROJET)

# Vérification
n = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY").filterDate("2023-01-01", "2025-01-01").size()
print("Images CHIRPS journalières en 2024 :", n.getInfo())

✅ Earth Engine initialisé
Images CHIRPS journalières en 2024 : 731


### 1.1 La notion à intégrer avant tout : serveur ≠ client

| Côté **serveur** (Google) | Côté **client** (votre machine) |
|---|---|
| `ee.Image`, `ee.Number`, `ee.List`, `ee.Dictionary` | `int`, `float`, `list`, `dict`, `DataFrame` |
| Évaluation **paresseuse** : rien n'est calculé avant qu'on demande le résultat | Immédiate |
| `.map()`, `.filter()`, `.reduce()` | boucles `for`, `if` |
| Passage au client : `.getInfo()`, `Export.*` | — |

```python
# ❌ MAUVAIS : une boucle Python sur une collection serveur
for i in range(coll.size().getInfo()):        # un aller-retour réseau par itération
    ee.Image(coll.toList(1000).get(i)).reduceRegion(...).getInfo()

# ✅ BON : tout reste sur le serveur, un seul aller-retour à la fin
coll.map(lambda img: ...).getInfo()
```

**Règle** : n'appelez `.getInfo()` **qu'une fois, à la toute fin**. Chaque appel est une requête
réseau synchrone et un point d'échec.

---

## 2. Les sources pluviométriques

| Identifiant | Résolution | Pas | Période | Nature |
|---|---|---|---|---|
| `UCSB-CHG/CHIRPS/DAILY` | 0,05° (~5,5 km) | jour | 1981 → | Satellite **+ stations au sol** |
| `ECMWF/ERA5_LAND/DAILY_AGGR` | 0,1° (~11 km) | jour | 1950 → | Réanalyse physique |
| `NASA/GPM_L3/IMERG_V07` | 0,1° | 30 min | 2000 → | Satellite, événements extrêmes |

### Comment choisir

**CHIRPS** est la référence en zone semi-aride, et donc pour le Maroc. Elle assimile des relevés
de stations au sol, ce qui corrige le biais des produits purement satellitaires. C'est le standard
des systèmes d'alerte précoce (FEWS NET, FAO).

**ERA5-Land** est physiquement cohérente et sans trou, mais lissée : sur un relief marqué comme
le Haut Atlas, elle sous-estime les contrastes. En revanche elle remonte à 1950 et fournit
simultanément température, vent, rayonnement.

> Aucune source n'est « la vérité ». En production, on valide contre les stations de la
> **Direction Générale de la Météorologie** et on documente le biais résiduel.

### Attention aux unités
CHIRPS est déjà en **mm/jour**. ERA5-Land est en **mètres** : il faut multiplier par 1000.
C'est l'erreur la plus fréquente, et elle passe inaperçue parce que le résultat reste plausible.

> **Extension à ce notebook (ajout local) :** dans la version d'origine, seules CHIRPS et
> ERA5-Land étaient codées ; IMERG figurait juste dans le tableau ci-dessus. Le moteur
> (`collection_pluie`, `analyser`, `synthese`, `tracer`) a été généralisé pour accepter
> n'importe quel pas de temps (`pas_minutes` dans `SOURCES`), et IMERG 30 min a été ajouté
> comme troisième option dans le menu déroulant de la section 7.2.
>
> **Quand choisir IMERG plutôt que CHIRPS** : pour un **épisode précis** (crue, orage, jour
> de coupure) où l'intensité infra-journalière compte — pas pour une climatologie longue
> durée (le volume de données explose : 48 pas/jour contre 1). Le garde-fou de la section 7.2
> le rappelle si la période choisie est trop longue pour ce pas de temps.

In [ ]:
SOURCES = {
    "CHIRPS (recommandé, 5.5 km)": {
        "id": "UCSB-CHG/CHIRPS/DAILY",
        "bande": "precipitation",
        "facteur": 1.0,               # déjà en mm/jour
        "echelle": 5566,              # résolution native en mètres
        "debut": "1981-01-01",
        "pas_minutes": 1440,          # 1 jour
    },
    "ERA5-Land (11 km, depuis 1950)": {
        "id": "ECMWF/ERA5_LAND/DAILY_AGGR",
        "bande": "total_precipitation_sum",
        "facteur": 1000.0,            # mètres -> millimètres
        "echelle": 11132,
        "debut": "1950-01-02",
        "pas_minutes": 1440,          # 1 jour
    },
    "IMERG (30 min, événements extrêmes)": {
        "id": "NASA/GPM_L3/IMERG_V07",
        "bande": "precipitation",
        "facteur": 0.5,                # mm/h -> mm par pas de 30 min (0,5 h)
        "echelle": 11132,
        "debut": "2000-06-01",
        "pas_minutes": 30,
    },
}

def collection_pluie(nom_source, debut, fin, roi):
    """Retourne une ImageCollection homogénéisée en mm PAR PAS DE TEMPS (bande 'prcp').

    Le pas dépend de la source (`pas_minutes` : 1440 pour un produit journalier,
    30 pour IMERG). `facteur` fait toujours passer l'unité native de l'image vers
    « mm reçus pendant ce pas » — c'est pour cela qu'IMERG (mm/h) utilise 0.5 :
    un mm/h moyenné sur une demi-heure donne 0.5 mm.
    """
    cfg = SOURCES[nom_source]
    coll = (ee.ImageCollection(cfg["id"])
              .filterDate(str(debut), str(fin))
              .filterBounds(roi)
              .select(cfg["bande"]))

    f = cfg["facteur"]
    def normaliser(img):
        return (img.multiply(f).rename("prcp")
                   .copyProperties(img, ["system:time_start"]))

    return coll.map(normaliser), cfg

print("Sources déclarées :", list(SOURCES))

---

## 3. La question méthodologique centrale : dans quel ordre réduire ?

« Le cumul, la moyenne, le max et le min de la pluie sur la zone et sur la période » est une
formulation **ambiguë**. Il y a deux dimensions à agréger — l'espace et le temps — et l'ordre
change le résultat.

### Bloc A — On agrège d'abord dans le temps, puis on regarde l'espace

On somme les images jour par jour → une **image unique** du cumul de la période.
On la réduit ensuite sur la zone.

- *Cumul moyen* : la lame d'eau moyenne reçue par la zone sur toute la période.
- *Cumul max* : le pixel le plus arrosé de la zone.
- *Cumul min* : le pixel le plus sec.

→ Ces statistiques décrivent l'**hétérogénéité spatiale** de la pluviométrie.
C'est ce qu'on veut pour comparer un versant au vent et un versant sous le vent.

### Bloc B — On agrège d'abord dans l'espace, puis on regarde le temps

Pour chaque jour, on calcule la moyenne sur la zone → une **série temporelle**.

- *Cumul* : la somme de la série (identique au cumul moyen du bloc A — bonne vérification).
- *Moyenne journalière* : intensité moyenne d'un jour.
- *Max journalier* : la journée la plus pluvieuse.
- *Min journalier* : presque toujours 0 en climat semi-aride, donc peu informatif seul.

→ Ces statistiques décrivent la **dynamique temporelle**. C'est ce qu'on veut pour repérer un
épisode extrême ou mesurer la concentration des pluies.

> **Le piège classique** : confondre « le pixel le plus arrosé de la zone » (bloc A) et
> « le jour le plus pluvieux en moyenne sur la zone » (bloc B). Ce sont deux maximums différents,
> et l'écart entre les deux est souvent d'un facteur 10.
>
> L'outil calcule et affiche **les deux**, explicitement étiquetés. Un tableau de résultats qui
> annonce « max : 87 mm » sans préciser lequel est inexploitable.

---

## 4. Combien de données Earth Engine traite-t-il ?

Le calcul est direct. Pour un raster :

```
volume traité  =  nb_pixels  ×  nb_images  ×  octets_par_pixel
```

- **`nb_pixels`** = surface de la zone dessinée ÷ (résolution native)²
- **`nb_images`** = nombre de jours de la période
- **`octets_par_pixel`** = 4 en `float32`, précision standard pour une variable continue

C'est le volume qu'Earth Engine **parcourt effectivement** pour répondre à la requête : chaque
pixel de chaque image de la période est lu, puis agrégé.

Ce qui revient vers votre machine tient en quelques dizaines de kilo-octets — la série et les
statistiques. Le rapport entre les deux est tout l'intérêt de la plateforme : **le calcul est
envoyé aux données, pas l'inverse.**

> La résolution native compte double : elle intervient **au carré** dans le nombre de pixels.
> Passer de CHIRPS (5,5 km) à un produit à 1 km multiplie le volume traité par **31**,
> à zone et période identiques.

In [ ]:
OCTETS_PAR_PIXEL = 4      # float32

def format_octets(n):
    for unite in ["o", "Ko", "Mo", "Go", "To"]:
        if abs(n) < 1024:
            return f"{n:,.1f} {unite}".replace(",", " ")
        n /= 1024
    return f"{n:,.1f} Po"

def estimer_volume(surface_m2, n_images, echelle_m):
    """Volume parcouru par Earth Engine pour cette zone et cette période."""
    pixels_par_image = surface_m2 / (echelle_m ** 2)
    observations = pixels_par_image * n_images
    return {
        "n_images": n_images,
        "pixels_par_image": pixels_par_image,
        "observations_pixel": observations,
        "volume_octets": observations * OCTETS_PAR_PIXEL,
    }

def afficher_volume(v, nom_source, surface_km2, echelle_m, libelle_pl="jours"):
    esp = lambda x: f"{x:,.0f}".replace(",", " ")
    print("=" * 62)
    print("  DONNÉES TRAITÉES PAR EARTH ENGINE")
    print("=" * 62)
    print(f"  Source              : {nom_source}")
    print(f"  Résolution native   : {esp(echelle_m)} m")
    print(f"  Zone sélectionnée   : {esp(surface_km2)} km²")
    print(f"  Images ({libelle_pl:<12s}) : {esp(v['n_images'])}")
    print("-" * 62)
    print(f"  Pixels par image    : {esp(v['pixels_par_image'])}")
    print(f"  Observations-pixel  : {esp(v['observations_pixel'])}")
    print(f"  Volume traité       : {format_octets(v['volume_octets'])}")
    print("=" * 62)

# Démonstration : plaine du Gharb (~9 800 km²) sur une année de CHIRPS
demo = estimer_volume(surface_m2=9_800e6, n_images=365, echelle_m=5566)
afficher_volume(demo, "CHIRPS (recommandé, 5.5 km)", 9_800, 5566)

---

## 5. Le moteur de calcul

Une seule fonction, entièrement côté serveur jusqu'au `getInfo()` final.

Points techniques à noter :

- **Réducteur combiné** : `mean().combine(minMax()).combine(stdDev())` calcule quatre statistiques
  en **un seul passage** sur les pixels. Beaucoup plus rapide que quatre `reduceRegion` successifs.
- **`scale` explicite** : toujours. Sans cela GEE choisit une valeur arbitraire et le résultat
  n'est pas reproductible.
- **`bestEffort=False`** : on refuse la dégradation silencieuse de résolution. Si le calcul est
  trop lourd, on veut une erreur, pas un résultat faux.
- **`tileScale`** : découpe le calcul en tuiles plus petites. À augmenter (4, 8, 16) en cas
  d'erreur mémoire.

In [ ]:
REDUCTEUR = (ee.Reducer.mean()
             .combine(ee.Reducer.minMax(), sharedInputs=True)
             .combine(ee.Reducer.stdDev(), sharedInputs=True))

def analyser(roi, debut, fin, nom_source="CHIRPS (recommandé, 5.5 km)", tile_scale=4):
    """Analyse pluviométrique complète. Un seul aller-retour serveur -> client."""
    coll, cfg = collection_pluie(nom_source, debut, fin, roi)
    echelle = cfg["echelle"]
    sub_journalier = cfg["pas_minutes"] < 1440
    format_date = "YYYY-MM-dd HH:mm" if sub_journalier else "YYYY-MM-dd"

    # ---- BLOC A : cumul temporel, puis statistiques spatiales ----
    # coll.sum() reste correct quel que soit le pas : chaque image porte déjà
    # "mm reçus pendant CE pas" (grâce à `facteur`), donc sommer les images
    # intègre bien la lame d'eau totale sur toute la période.
    img_cumul = coll.sum().rename("cumul")
    stats_spatiales = img_cumul.reduceRegion(
        reducer=REDUCTEUR, geometry=roi, scale=echelle,
        maxPixels=1e11, bestEffort=False, tileScale=tile_scale)

    # ---- BLOC B : moyenne spatiale par pas de temps, puis série temporelle ----
    def par_pas(img):
        v = img.reduceRegion(
            reducer=REDUCTEUR, geometry=roi, scale=echelle,
            maxPixels=1e11, bestEffort=False, tileScale=tile_scale)
        return ee.Feature(None, {
            "date":     img.date().format(format_date),
            "moyenne":  v.get("prcp_mean"),
            "min":      v.get("prcp_min"),
            "max":      v.get("prcp_max"),
            "ecarttype": v.get("prcp_stdDev"),
        })

    serie = ee.FeatureCollection(coll.map(par_pas))

    # ---- Métadonnées ----
    meta = ee.Dictionary({
        "n_images":   coll.size(),
        "surface_m2": roi.area(maxError=100),
    })

    # ---- UN SEUL aller-retour ----
    paquet = ee.Dictionary({
        "spatial": stats_spatiales,
        "serie":   serie,
        "meta":    meta,
    }).getInfo()

    # ---- Mise en forme côté client ----
    lignes = [f["properties"] for f in paquet["serie"]["features"]]
    df = (pd.DataFrame(lignes)
            .assign(date=lambda d: pd.to_datetime(d["date"]))
            .sort_values("date").reset_index(drop=True))
    for c in ["moyenne", "min", "max", "ecarttype"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df["cumul"] = df["moyenne"].cumsum()

    return {
        "df": df,
        "spatial": paquet["spatial"],
        "meta": paquet["meta"],
        "cfg": cfg,
        "img_cumul": img_cumul,
        "source": nom_source,
        "libelle_pas": "pas de 30 min" if sub_journalier else "jour",
        "libelle_pas_pluriel": "pas de 30 min" if sub_journalier else "jours",
    }

print("Moteur d'analyse prêt.")

---

## 6. La restitution : tableau de synthèse et graphiques

In [ ]:
def synthese(res):
    """Tableau des quatre statistiques, sous leurs deux formes.

    Généralisé au pas de temps de la source (`res["libelle_pas"]`) : un jour
    pour CHIRPS/ERA5-Land, un pas de 30 min pour IMERG.
    """
    df, sp = res["df"], res["spatial"]
    libelle = res["libelle_pas"]
    libelle_pl = res["libelle_pas_pluriel"]
    cumul_total = float(df["moyenne"].sum())

    bloc_a = {
        "Cumul moyen sur la zone":   sp.get("cumul_mean"),
        "Cumul du pixel le + arrosé": sp.get("cumul_max"),
        "Cumul du pixel le + sec":    sp.get("cumul_min"),
        "Écart-type spatial":         sp.get("cumul_stdDev"),
    }
    bloc_b = {
        "Cumul de la période":                       cumul_total,
        f"Moyenne par {libelle}":                    float(df["moyenne"].mean()),
        f"{libelle.capitalize()} le plus pluvieux":  float(df["moyenne"].max()),
        f"{libelle.capitalize()} le moins pluvieux": float(df["moyenne"].min()),
        f"Pic ponctuel (1 pixel, 1 {libelle})":      float(df["max"].max()),
        f"{libelle_pl.capitalize()} de pluie (> 1 mm)":    int((df["moyenne"] > 1).sum()),
        f"{libelle_pl.capitalize()} secs (< 0,1 mm)":      int((df["moyenne"] < 0.1).sum()),
    }
    return bloc_a, bloc_b, cumul_total

def afficher_synthese(res):
    a, b, cumul_total = synthese(res)
    df = res["df"]
    libelle_pl = res["libelle_pas_pluriel"]

    print("=" * 68)
    print(f"  RÉSULTATS — {res['source']}")
    print(f"  Période : {df['date'].min():%d/%m/%Y %H:%M} → {df['date'].max():%d/%m/%Y %H:%M}"
          f"  ({len(df)} {libelle_pl})")
    print("=" * 68)
    print("\n  BLOC A — Cumul de la période, distribution dans l'espace (mm)")
    for k, v in a.items():
        print(f"    {k:32s} {float(v):10.1f}")
    print("\n  BLOC B — Moyenne sur la zone, dynamique dans le temps (mm)")
    for k, v in b.items():
        fmt = f"{v:10.1f}" if isinstance(v, float) else f"{v:10d}"
        print(f"    {k:32s} {fmt}")

    ecart = abs(cumul_total - float(a["Cumul moyen sur la zone"]))
    print(f"\n  Contrôle de cohérence A/B : écart = {ecart:.3f} mm", end="")
    print("  ✅" if ecart < 1 else "  ⚠️ vérifier l'échelle de réduction")

def tracer(res):
    df = res["df"]
    libelle = res["libelle_pas"]
    pas_minutes = res["cfg"]["pas_minutes"]
    largeur_barre = max(pas_minutes / 1440 * 0.9, 0.01)   # en jours, unité de l'axe des dates

    fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=False)

    # 1. Série par pas de temps + enveloppe spatiale min-max
    ax = axes[0]
    ax.fill_between(df["date"], df["min"], df["max"], color="#aed6f1", alpha=.6,
                    label="Étendue spatiale (min-max des pixels)")
    ax.bar(df["date"], df["moyenne"], width=largeur_barre, color="#1f618d",
           label="Moyenne sur la zone")
    ax.set_ylabel(f"Pluie par {libelle} (mm)")
    ax.set_title(f"Pluviométrie par {libelle} — {res['source']}", fontweight="bold")
    ax.legend(); ax.grid(alpha=.3)

    # 2. Courbe de cumul
    ax = axes[1]
    ax.plot(df["date"], df["cumul"], lw=2.4, color="#154360")
    ax.fill_between(df["date"], 0, df["cumul"], color="#154360", alpha=.15)
    ax.set_ylabel("Cumul (mm)")
    ax.set_title(f"Cumul progressif — total : {df['cumul'].iloc[-1]:.1f} mm", fontweight="bold")
    ax.grid(alpha=.3)

    # 3. Profil mensuel (resample("MS").sum() reste correct quel que soit le pas :
    #    la colonne "moyenne" est déjà en mm reçus pendant le pas de la source)
    ax = axes[2]
    mens = df.set_index("date")["moyenne"].resample("MS").sum()
    ax.bar(mens.index, mens.values, width=20, color="#2874a6", alpha=.85)
    ax.axhline(mens.mean(), color="k", ls="--", lw=1,
               label=f"Moyenne mensuelle : {mens.mean():.1f} mm")
    ax.set_ylabel("Cumul mensuel (mm)"); ax.set_xlabel("Date")
    ax.legend(); ax.grid(alpha=.3, axis="y")

    plt.tight_layout(); plt.show()

print("Fonctions de restitution prêtes.")

---

## 7. L'application

### 7.1 La carte — dessinez votre zone

Utilisez les outils de dessin dans la barre latérale gauche : **rectangle** ou **polygone**.
La dernière forme dessinée est celle qui sera analysée.

> Commencez modestement. Une zone de la taille d'une province sur un an s'analyse en quelques
> secondes ; le Maroc entier sur trente ans demandera un export asynchrone (module 9).

In [ ]:
Carte = geemap.Map(center=[31.8, -7.1], zoom=5)
Carte.add_basemap("HYBRID")


Carte

Map(center=[31.8, -7.1], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …

In [ ]:
def zone_dessinee(carte, defaut=None):
    """Récupère la dernière géométrie dessinée, avec repli sur une zone par défaut."""
    for attr in ("user_roi", "draw_last_feature"):
        obj = getattr(carte, attr, None)
        if obj is None:
            continue
        try:
            return obj.geometry() if hasattr(obj, "geometry") else obj
        except Exception:
            return obj
    if defaut is not None:
        print("ℹ️  Aucune zone dessinée — utilisation de la zone par défaut.")
        return defaut
    raise ValueError("Dessinez d'abord une zone sur la carte.")

# Zone de repli : bassin du Sebou / plaine du Gharb
ZONE_DEFAUT = ee.Geometry.Rectangle([-6.6, 34.0, -5.2, 34.9])
print("Fonction de récupération de la zone prête.")

Fonction de récupération de la zone prête.


### 7.2 Le panneau de commande

In [ ]:
w_source = widgets.Dropdown(options=list(SOURCES), value=list(SOURCES)[0],
                            description="Source :", layout=widgets.Layout(width="420px"),
                            style={"description_width": "110px"})
w_debut  = widgets.DatePicker(value=date(2024, 9, 1), description="Début :",
                              style={"description_width": "110px"})
w_fin    = widgets.DatePicker(value=date(2025, 8, 31), description="Fin :",
                              style={"description_width": "110px"})
w_carte  = widgets.Checkbox(value=True, description="Ajouter la carte du cumul")
w_bouton = widgets.Button(description="Analyser", button_style="primary",
                          icon="play", layout=widgets.Layout(width="180px", height="38px"))
w_sortie = widgets.Output()

RESULTAT = {}          # conserve le dernier résultat pour les cellules suivantes

# Au-delà de ce nombre de pas, un seul .getInfo() synchrone devient risqué
# (timeout / mémoire) : mieux vaut réduire la période ou passer par Export (module 9).
MAX_PAS_SYNCHRONE = 4000

def au_clic(_):
    with w_sortie:
        clear_output(wait=True)
        d, f = w_debut.value, w_fin.value

        # --- Validation des saisies ---
        if d is None or f is None:
            print("⛔ Renseignez les deux dates."); return
        if f <= d:
            print("⛔ La date de fin doit être postérieure à la date de début."); return
        cfg_source = SOURCES[w_source.value]
        mini = date.fromisoformat(cfg_source["debut"])
        if d < mini:
            print(f"⛔ {w_source.value} commence le {mini}."); return

        n_pas_estime = (f - d).days * (1440 // cfg_source["pas_minutes"])
        if n_pas_estime > MAX_PAS_SYNCHRONE:
            print(f"⚠️  Période trop longue pour {w_source.value} : environ {n_pas_estime:,} pas "
                  f"estimés (limite {MAX_PAS_SYNCHRONE:,}).".replace(",", " "))
            print("   Réduisez la période, ou passez par un export asynchrone (module 9).")
            if cfg_source["pas_minutes"] < 1440:
                print("   IMERG (30 min) est prévu pour des épisodes courts (jours/semaines),")
                print("   pas pour des climatologies pluriannuelles — utilisez CHIRPS pour ça.")
            return

        try:
            roi = zone_dessinee(Carte, defaut=ZONE_DEFAUT)
        except ValueError as e:
            print("⛔", e); return

        print("Calcul en cours sur les serveurs Earth Engine…\n")
        try:
            res = analyser(roi, d, f, w_source.value)
        except ee.EEException as e:
            print("⛔ Erreur Earth Engine :", e)
            print("   Piste : augmenter tileScale, réduire la zone ou la période.")
            return

        if res["df"].empty:
            print("⛔ Aucune image sur cette période et cette zone."); return

        RESULTAT.clear(); RESULTAT.update(res)
        RESULTAT["roi"] = roi

        # 1) Volume de données traité
        surface = float(res["meta"]["surface_m2"])
        vol = estimer_volume(surface, int(res["meta"]["n_images"]), res["cfg"]["echelle"])
        afficher_volume(vol, res["source"], surface / 1e6, res["cfg"]["echelle"],
                        libelle_pl=res["libelle_pas_pluriel"])

        # 2) Statistiques
        print(); afficher_synthese(res)

        # 3) Graphiques
        print(); tracer(res)

        # 4) Carte du cumul
        if w_carte.value:
            cumul_max = float(res["spatial"]["cumul_max"] or 100)
            vis = {"min": 0, "max": cumul_max,
                   "palette": ["#f7fbff", "#c6dbef", "#6baed6", "#2171b5", "#08306b"]}
            Carte.addLayer(res["img_cumul"].clip(roi), vis,
                           f"Cumul {d} → {f}")
            Carte.add_colorbar(vis, label="Cumul (mm)", orientation="horizontal")
            print("🗺️  Couche ajoutée à la carte — remontez pour la visualiser.")

w_bouton.on_click(au_clic)

display(widgets.VBox([
    widgets.HTML("<h3>Analyseur pluviométrique</h3>"
                 "<p>Dessinez une zone sur la carte ci-dessus, choisissez la période, "
                 "puis cliquez sur <b>Analyser</b>.</p>"),
    w_source,
    widgets.HBox([w_debut, w_fin]),
    w_carte,
    w_bouton,
    w_sortie,
]))

---

## 8. Exploiter le résultat

Après un clic sur « Analyser », le dictionnaire `RESULTAT` contient tout :
`RESULTAT["df"]` (la série journalière), `RESULTAT["spatial"]` (les statistiques spatiales),
`RESULTAT["img_cumul"]` (l'image du cumul), `RESULTAT["roi"]` (la géométrie).

In [ ]:
if RESULTAT:
    df = RESULTAT["df"]
    libelle_pl = RESULTAT["libelle_pas_pluriel"]

    # Table mensuelle exploitable directement (nom générique : "n_pas" plutôt que
    # "jours", car pour IMERG chaque ligne est un pas de 30 min, pas un jour)
    mensuel = (df.set_index("date")["moyenne"]
                 .resample("MS")
                 .agg(cumul="sum", moyenne_pas="mean", max_pas="max", n_pas="size")
                 .round(2))
    display(mensuel)

    # Export CSV
    from pathlib import Path
    OUT = Path("sorties"); OUT.mkdir(exist_ok=True)
    horodatage = datetime.now().strftime("%Y%m%d_%H%M")
    df.to_csv(OUT / f"pluie_serie_{horodatage}.csv", index=False)
    mensuel.to_csv(OUT / f"pluie_mensuelle_{horodatage}.csv")
    print(f"Série au pas de temps : {libelle_pl} ({len(df)} lignes)")
    print("Fichiers écrits dans", OUT.resolve())
else:
    print("Lancez d'abord une analyse.")

### 8.1 Pas de temps remarquables (jour ou demi-heure, selon la source)

In [ ]:
if RESULTAT:
    df = RESULTAT["df"]
    libelle_pl = RESULTAT["libelle_pas_pluriel"]
    print(f"Dix {libelle_pl} les plus pluvieux (moyenne sur la zone) :\n")
    top = df.nlargest(10, "moyenne")[["date", "moyenne", "max", "ecarttype"]].copy()
    top.columns = ["date", "moyenne_zone_mm", "pixel_max_mm", "ecart_type_mm"]
    top["ratio_pic_moyenne"] = (top["pixel_max_mm"] / top["moyenne_zone_mm"]).round(1)
    display(top.round(2))
    print("\nUn ratio élevé signale un épisode très localisé (orage convectif) ;")
    print("un ratio proche de 1 signale une pluie stratiforme homogène sur la zone.")
else:
    print("Lancez d'abord une analyse.")

---

## 9. Passer à l'échelle : les exports asynchrones

L'outil ci-dessus utilise `.getInfo()`, qui est **synchrone** et plafonné.
Au-delà de quelques milliers d'images, ou pour une zone très étendue, il faut passer par
`Export.*` : la tâche part sur les serveurs Google et vous récupérez le résultat plus tard.

Suivi des tâches : https://code.earthengine.google.com/tasks

In [ ]:
if RESULTAT:
    roi = RESULTAT["roi"]

    # --- Raster du cumul, en projection marocaine ---
    tache_tif = ee.batch.Export.image.toDrive(
        image=RESULTAT["img_cumul"].toFloat(),
        description="HCP_cumul_pluie",
        folder="GEE_HCP",
        fileNamePrefix=f"cumul_pluie_{datetime.now():%Y%m%d}",
        region=roi,
        scale=RESULTAT["cfg"]["echelle"],
        crs="EPSG:26191",            # Lambert Conforme Conique Maroc (Merchich zone I)
        maxPixels=1e11,
        fileFormat="GeoTIFF",
    )

    print("Tâche définie. Décommentez pour lancer :")
    print("  tache_tif.start(); print(tache_tif.status())")
    # tache_tif.start()
else:
    print("Lancez d'abord une analyse.")

### 9.1 Automatisation par compte de service

Pour une exécution planifiée (Airflow, Celery, cron), l'authentification interactive est
impossible.

```python
CLE    = "/chemin/securise/hcp-gee-sa.json"     # jamais dans le dépôt Git
COMPTE = "gee-runner@hcp-geospatial-2026.iam.gserviceaccount.com"

creds = ee.ServiceAccountCredentials(COMPTE, CLE)
ee.Initialize(creds, project="hcp-geospatial-2026")
```

- Clé montée en secret (Docker secret, Vault), jamais versionnée.
- Rôle Cloud minimal : `Earth Engine Resource Viewer`, plus `Storage Object Admin` si export GCS.
- **Quotas** : environ 3 000 requêtes/minute et 40 tâches d'export simultanées.
  Découper les traitements longs par année ou par région.

### Erreurs fréquentes

| Message | Cause | Remède |
|---|---|---|
| `User memory limit exceeded` | Réduction trop lourde | Augmenter `tileScale` (4, 8, 16), découper la période |
| `Computation timed out` | Chaîne d'opérations trop longue | Passer par `Export.*` |
| `Too many concurrent aggregations` | Trop de `.getInfo()` | Regrouper en un seul appel |
| Valeurs `null` | Zone hors emprise, ou aucun pixel valide à cette `scale` | Vérifier le CRS et la géométrie |
| Résultats variables d'une exécution à l'autre | `bestEffort=True` | Fixer `scale` et `maxPixels` explicitement |

---

## 10. Interpréter — et se méfier

**Une moyenne sur une grande zone hétérogène ne veut pas dire grand-chose.** Si votre polygone
couvre à la fois le Rif et la plaine du Gharb, le cumul moyen ne décrit aucun des deux.
L'écart-type spatial du bloc A est là pour vous alerter : s'il approche la moyenne,
il faut découper la zone.

**Les produits satellitaires sous-estiment les pluies orographiques.** En montagne, CHIRPS et
ERA5-Land manquent une partie des précipitations liées au relief. Ne comparez pas un versant
atlasique à une plaine sans vérifier contre les stations.

**Le cumul brut n'est pas un indicateur de sécheresse.** 200 mm est excédentaire à Ouarzazate et
catastrophique à Tanger. Il faut standardiser contre une **normale climatologique** — c'est le
principe du SPI, objet de l'exercice 4.

**La répartition compte autant que le total.** 300 mm en trois épisodes ruissellent ;
300 mm répartis sur quarante jours s'infiltrent. Le nombre de jours de pluie et le ratio
pic/moyenne du §8.1 sont souvent plus informatifs que le cumul lui-même.

---

## 11. Exercices

### Exercice 1 — Comparaison de sources *(facile)*
Sur la même zone et la même période, lancez l'analyse avec CHIRPS puis ERA5-Land.
Superposez les deux courbes de cumul. **Question** : l'écart est-il constant, ou dépend-il
de l'intensité des épisodes ?

### Exercice 2 — Effet de l'emprise *(facile)*
Dessinez trois zones emboîtées : un bassin versant, sa province, sa région.
Comparez l'écart-type spatial. **Question** : à partir de quelle taille la moyenne
cesse-t-elle d'être interprétable ?

### Exercice 3 — Indice d'extrêmes *(intermédiaire)*
Ajoutez au tableau de synthèse les indices **ETCCDI** : `Rx1day` (max sur 1 jour),
`Rx5day` (max sur 5 jours glissants), `CDD` (plus longue série de jours secs consécutifs),
`R20mm` (nombre de jours à plus de 20 mm).

### Exercice 4 — Anomalie standardisée *(intermédiaire)*
Ajoutez un second calcul comparant le cumul de la période à la **normale 1991-2020** du même
mois. Affichez l'écart en pourcentage et en écarts-types. **Question** : pourquoi une simple
anomalie standardisée est-elle une approximation du SPI, et dans quels cas l'écart devient-il
gênant ?

### Exercice 5 — Découpage administratif *(intermédiaire)*
Remplacez le dessin libre par un menu déroulant listant les **12 régions officielles**,
alimenté par un asset GEE téléversé depuis le shapefile du HCP. Conservez les codes officiels
comme propriétés — c'est la clé de jointure avec le RGPH et la comptabilité régionale.

### Exercice 6 — Sensibilité du volume traité *(avancé)*
Lancez la même période sur trois zones de tailles croissantes, puis la même zone sur trois
périodes croissantes. Tracez le volume traité dans chaque cas. **Question** : les deux relations
sont linéaires — mais que devient le volume si vous passez à une source à 1 km de résolution,
et pourquoi cet effet-là est-il quadratique ?

### Exercice 7 — Pluie et population *(avancé)*
Pondérez le cumul par la population en croisant avec `JRC/GHSL/P2023A/GHS_POP` (grille 100 m).
Vous obtenez la pluviométrie **vécue par les habitants** plutôt que par les kilomètres carrés.
**Question** : sur une région comme Marrakech-Safi, l'écart entre les deux moyennes est-il
significatif, et que dit-il de la localisation du peuplement ?

---

## Ressources

- **Catalogue GEE** : https://developers.google.com/earth-engine/datasets
- **Documentation Python** : https://developers.google.com/earth-engine/guides/python_install
- **geemap** (Qiusheng Wu) : https://geemap.org
- **CHIRPS** — Funk et al. (2015), *Scientific Data* : la publication de référence du produit
- **ETCCDI** : définitions normalisées des indices d'extrêmes climatiques
- **UN-GGIM** — *Earth Observations for Official Statistics* ; travaux du groupe de villes
  de Kigali sur les données satellitaires en statistique publique